In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.analytic import ProbabilityOfImprovement
import copy

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M25.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7708525187757389, 'n_it': 0.39743752977939806}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0, 1])),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": ProbabilityOfImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7):
        IterationClient = copy.deepcopy(client)
        IterationTrials = {}
        for __ in range(3):
            SampleTrial = IterationClient.get_next_trials(max_trials=1)
            for trial_index, parameters in SampleTrial.items():
                IterationTrials[trial_index]=parameters
                s1 = parameters["s1"]
                s2 = parameters["s2"]
                b1 = parameters["b1"]
                result = IterationClient.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
                raw_data = {metric_name: result}
                IterationClient.complete_trial(trial_index=trial_index, raw_data=raw_data)
        for trial_index, parameters in IterationTrials.items():
            client.attach_trial(parameters=parameters)
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            raw_data = {metric_name: result}
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(np.array(client.summarize().t1).tolist()[0:27]))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.76357473223122

Trial 1 =========================================
13.377301674049692

Trial 2 =========================================
17.49926487032498

Trial 3 =========================================
17.908110541480063

Trial 4 =========================================
15.04318165369638

Trial 5 =========================================
13.886199285313038

Trial 6 =========================================
17.90815248398914

Trial 7 =========================================
13.900796801820045

Trial 8 =========================================
18.23900520489609

Trial 9 =========================================
13.72303774562841

Trial 10 =========================================
13.793396116884091

Trial 11 =========================================
13.236241234557351

Trial 12 =========================================
13.798074496084869

Trial 13 =========================================
13.675219704448274

Trial 14 =============

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.293313222835916
Avg = 15.499160220580505
Std = 2.00504656431767


In [7]:
print(y_max_arr.tolist())

[15.76357473223122, 13.377301674049692, 17.49926487032498, 17.908110541480063, 15.04318165369638, 13.886199285313038, 17.90815248398914, 13.900796801820045, 18.23900520489609, 13.72303774562841, 13.793396116884091, 13.236241234557351, 13.798074496084869, 13.675219704448274, 13.83539359293859, 13.796864024831647, 13.359540617775794, 18.216920074063765, 13.851475254485043, 18.064500211055552, 18.2425035906116, 18.18217630225925, 13.797699216169704, 13.856710226393258, 14.542614718847313, 17.9301151271128, 13.887308404756078, 13.806872860619102, 13.79221327259268, 17.134048258702485, 18.158661302716503, 16.025871459578358, 13.872362324590569, 17.35900258104428, 17.271181862158127, 13.741510256928983, 17.714151188564855, 18.02023032769206, 13.305725936222526, 13.869100928105821, 18.254357488362487, 13.871367731081387, 17.127044997746594, 13.909295513902816, 13.905537186542746, 13.379742023082667, 13.304177800457731, 13.906471271657344, 18.07712185455235, 13.611515181043545, 13.069680075153

In [8]:
# filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestswGPModel/DataGenerated/normal_PI_9_27_3.pkl"
# latestdf = pd.DataFrame(y_max_arr)
# pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    13.893656
1    13.907002
2    18.252604
3    18.039412
4    13.784970
..         ...
295  13.797800
296  17.360178
297  13.368096
298  17.631773
299  13.714705

[300 rows x 1 columns]
